In [1]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset(
    "csv",
    data_files="processed_reviews_2.csv",
    encoding="latin-1"
)

C:\Users\LEGEND\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 315 examples [00:00, 9430.72 examples/s]


In [2]:
# Format the datase
def format_example(example):
    return {
        "text": f"Game: {example['app_name']}\nInput: {example['input']}\nOutput: {example['output']}"
    }

dataset = dataset.map(format_example)

Map: 100%|██████████| 315/315 [00:00<00:00, 7296.09 examples/s]


In [3]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    
    tokens["labels"] = tokens["input_ids"].copy()
    
    return tokens

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 315/315 [00:00<00:00, 2223.20 examples/s]


In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.1)

In [ ]:
dataset = dataset.remove_columns(["app_name", "input", "output", "text"])

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./tinyllama-game-model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    fp16=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [ ]:
trainer.train()

In [ ]:
model.eval()

In [ ]:
prompt = "Game: Minecraft\nInput: a sandbox survival game\nOutput:"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# save the model
model.save_pretrained("my_game_model")
tokenizer.save_pretrained("my_game_model")

In [ ]:
# reload the model
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("my_game_model").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("my_game_model")

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

In [ ]:
predictions = []
references = []

model.eval()

for example in dataset["test"]:
    prompt = f"Game: {example['app_name']}\nInput: {example['input']}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    prediction = generated_text.split("Output:")[-1].strip()
    
    predictions.append(prediction)
    references.append(example["output"])

In [ ]:
results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
model.train()
model.gradient_checkpointing_enable()

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_finetune_model",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    num_train_epochs=3,
    learning_rate=2e-5,

    fp16=False,

    logging_steps=10,
    save_steps=100,

    do_eval=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

In [ ]:
trainer.train()